In [0]:
%sql
-- Catalog
CREATE CATALOG IF NOT EXISTS h2;
-- Schemas
CREATE SCHEMA IF NOT EXISTS h2.raw_entsoe;
CREATE SCHEMA IF NOT EXISTS h2.raw_era5;
CREATE SCHEMA IF NOT EXISTS h2.raw_h2stations;
CREATE SCHEMA IF NOT EXISTS h2.silver;
CREATE SCHEMA IF NOT EXISTS h2.gold;
-- Volume (must be in a schema)
CREATE VOLUME IF NOT EXISTS h2.raw_entsoe.landing;
CREATE VOLUME IF NOT EXISTS h2.raw_era5.landing;
CREATE VOLUME IF NOT EXISTS h2.raw_h2stations.landing;

In [0]:
%sql
-- Verify files in each landing volume
LIST '/Volumes/h2/raw_entsoe/landing';

In [0]:
%sql
LIST '/Volumes/h2/raw_era5/landing';

In [0]:
%sql
LIST '/Volumes/h2/raw_h2stations/landing';

In [0]:
%sql
-- Entso-e prices 2020
CREATE OR REPLACE TABLE h2.raw_entsoe.prices_2020_raw USING DELTA AS
SELECT 
  `MTU (CET/CEST)` AS MTU_CET_CEST,
  Area,
  Sequence,
  `Day-ahead Price (EUR/MWh)` AS Day_ahead_Price_EUR_MWh,
  `Intraday Period (CET/CEST)` AS Intraday_Period_CET_CEST,
  `Intraday Price (EUR/MWh)` AS Intraday_Price_EUR_MWh,
  input_file_name() AS source_file,
  current_timestamp() AS ingested_at_utc
FROM read_files('/Volumes/h2/raw_entsoe/landing/entsoe_nl_2020_prices.csv', format => 'csv', header => true);

In [0]:
%sql
-- Entso-e prices 2021
CREATE OR REPLACE TABLE h2.raw_entsoe.prices_2021_raw USING DELTA AS
SELECT 
  `MTU (CET/CEST)` AS MTU_CET_CEST,
  Area,
  Sequence,
  `Day-ahead Price (EUR/MWh)` AS Day_ahead_Price_EUR_MWh,
  `Intraday Period (CET/CEST)` AS Intraday_Period_CET_CEST,
  `Intraday Price (EUR/MWh)` AS Intraday_Price_EUR_MWh,
  input_file_name() AS source_file,
  current_timestamp() AS ingested_at_utc
FROM read_files('/Volumes/h2/raw_entsoe/landing/entsoe_nl_2021_prices.csv', format => 'csv', header => true);

In [0]:
%sql
-- Entso-e load 2020
CREATE OR REPLACE TABLE h2.raw_entsoe.load_2020_raw USING DELTA AS
SELECT 
  `MTU (CET/CEST)` AS MTU_CET_CEST,
  Area,
  `Actual Total Load (MW)` AS Actual_Total_Load_MW,
  `Day-ahead Total Load Forecast (MW)` AS Day_ahead_Total_Load_Forecast_MW,
  input_file_name() AS source_file,
  current_timestamp() AS ingested_at_utc
FROM read_files('/Volumes/h2/raw_entsoe/landing/entsoe_nl_2020_load.csv', format => 'csv', header => true);

-- Entso-e load 2021
CREATE OR REPLACE TABLE h2.raw_entsoe.load_2021_raw USING DELTA AS
SELECT 
  `MTU (CET/CEST)` AS MTU_CET_CEST,
  Area,
  `Actual Total Load (MW)` AS Actual_Total_Load_MW,
  `Day-ahead Total Load Forecast (MW)` AS Day_ahead_Total_Load_Forecast_MW,
  input_file_name() AS source_file,
  current_timestamp() AS ingested_at_utc
FROM read_files('/Volumes/h2/raw_entsoe/landing/entsoe_nl_2021_load.csv', format => 'csv', header => true);

-- Entso-e generation 2020
CREATE OR REPLACE TABLE h2.raw_entsoe.generation_2020_raw USING DELTA AS
SELECT 
  `MTU (CET/CEST)` AS MTU_CET_CEST,
  Area,
  `Production Type` AS Production_Type,
  `Generation (MW)` AS Generation_MW,
  input_file_name() AS source_file,
  current_timestamp() AS ingested_at_utc
FROM read_files('/Volumes/h2/raw_entsoe/landing/entsoe_nl_2020_generation.csv', format => 'csv', header => true);

-- Entso-e generation 2021
CREATE OR REPLACE TABLE h2.raw_entsoe.generation_2021_raw USING DELTA AS
SELECT 
  `MTU (CET/CEST)` AS MTU_CET_CEST,
  Area,
  `Production Type` AS Production_Type,
  `Generation (MW)` AS Generation_MW,
  input_file_name() AS source_file,
  current_timestamp() AS ingested_at_utc
FROM read_files('/Volumes/h2/raw_entsoe/landing/entsoe_nl_2021_generation.csv', format => 'csv', header => true);

-- ERA5 weather 2020
CREATE OR REPLACE TABLE h2.raw_era5.weather_2020_raw USING DELTA AS
SELECT *, input_file_name() AS source_file, current_timestamp() AS ingested_at_utc
FROM read_files('/Volumes/h2/raw_era5/landing/era5_nl_2020_hourly.csv', format => 'csv', header => true);

-- ERA5 weather 2021
CREATE OR REPLACE TABLE h2.raw_era5.weather_2021_raw USING DELTA AS
SELECT *, input_file_name() AS source_file, current_timestamp() AS ingested_at_utc
FROM read_files('/Volumes/h2/raw_era5/landing/era5_nl_2021_hourly.csv', format => 'csv', header => true);

-- H2 stations NL
CREATE OR REPLACE TABLE h2.raw_h2stations.stations_nl_raw USING DELTA AS
SELECT *, input_file_name() AS source_file, current_timestamp() AS ingested_at_utc
FROM read_files('/Volumes/h2/raw_h2stations/landing/h2stations_nl.csv', format => 'csv', header => true);

In [0]:
%sql
OPTIMIZE h2.raw_entsoe.prices_2020_raw;
   OPTIMIZE h2.raw_entsoe.prices_2021_raw;
   OPTIMIZE h2.raw_entsoe.load_2020_raw;
   OPTIMIZE h2.raw_entsoe.load_2021_raw;
   OPTIMIZE h2.raw_entsoe.generation_2020_raw;
   OPTIMIZE h2.raw_entsoe.generation_2021_raw;
   OPTIMIZE h2.raw_era5.weather_2020_raw;
   OPTIMIZE h2.raw_era5.weather_2021_raw;
   OPTIMIZE h2.raw_h2stations.stations_nl_raw;

In [0]:
%sql
CREATE OR REPLACE VIEW h2.raw_entsoe.prices_all_raw AS
SELECT * FROM h2.raw_entsoe.prices_2020_raw
UNION ALL
SELECT * FROM h2.raw_entsoe.prices_2021_raw;

CREATE OR REPLACE VIEW h2.raw_entsoe.load_all_raw AS
SELECT * FROM h2.raw_entsoe.load_2020_raw
UNION ALL
SELECT * FROM h2.raw_entsoe.load_2021_raw;

CREATE OR REPLACE VIEW h2.raw_entsoe.generation_all_raw AS
SELECT * FROM h2.raw_entsoe.generation_2020_raw
UNION ALL
SELECT * FROM h2.raw_entsoe.generation_2021_raw;

CREATE OR REPLACE VIEW h2.raw_era5.weather_all_raw AS
SELECT * FROM h2.raw_era5.weather_2020_raw
UNION ALL
SELECT * FROM h2.raw_era5.weather_2021_raw;

In [0]:
%sql
-- 2) Row-count checks (separate, no cross-schema union needed)
SELECT
  (SELECT COUNT(*) FROM h2.raw_entsoe.load_all_raw) AS load_rows,
  (SELECT COUNT(*) FROM h2.raw_entsoe.generation_all_raw) AS generation_rows,
  (SELECT COUNT(*) FROM h2.raw_era5.weather_all_raw) AS weather_rows,
  (SELECT COUNT(*) FROM h2.raw_h2stations.stations_nl_raw) AS stations_rows,
  (SELECT COUNT(*) FROM h2.raw_entsoe.prices_all_raw) AS prices_rows;

In [0]:
%sql
-- 3) Schema checks
-- DESCRIBE TABLE h2.raw_entsoe.prices_all_raw;
-- DESCRIBE TABLE h2.raw_entsoe.load_all_raw;
-- DESCRIBE TABLE h2.raw_entsoe.generation_all_raw;
-- DESCRIBE TABLE h2.raw_era5.weather_all_raw;
-- DESCRIBE TABLE h2.raw_h2stations.stations_nl_raw;

In [0]:
from pyspark.sql import functions as F
import re
# =========================
# CONFIG
# =========================
CATALOG = "h2"
# raw tables
PRICES_2020 = f"{CATALOG}.raw_entsoe.prices_2020_raw"
PRICES_2021 = f"{CATALOG}.raw_entsoe.prices_2021_raw"
LOAD_2020 = f"{CATALOG}.raw_entsoe.load_2020_raw"
LOAD_2021 = f"{CATALOG}.raw_entsoe.load_2021_raw"
GEN_2020 = f"{CATALOG}.raw_entsoe.generation_2020_raw"
GEN_2021 = f"{CATALOG}.raw_entsoe.generation_2021_raw"
ERA5_2020 = f"{CATALOG}.raw_era5.weather_2020_raw"
ERA5_2021 = f"{CATALOG}.raw_era5.weather_2021_raw"
STATIONS_RAW = f"{CATALOG}.raw_h2stations.stations_nl_raw"
# silver outputs
T_PRICES_CLEAN = f"{CATALOG}.silver.entsoe_prices_clean"
T_LOAD_CLEAN = f"{CATALOG}.silver.entsoe_load_clean"
T_GEN_BY_TYPE = f"{CATALOG}.silver.entsoe_generation_hourly_by_type"
T_GEN_WIDE = f"{CATALOG}.silver.entsoe_generation_hourly_wide"
T_ERA5_CLEAN = f"{CATALOG}.silver.weather_clean"
T_STATIONS_CLEAN = f"{CATALOG}.silver.h2stations_clean"
T_TRAINING_BASE = f"{CATALOG}.silver.h2_training_base"

In [0]:
from pyspark.sql import functions as F
import re

def parse_mtu_start(col_name: str):
    return F.expr(f"""
      coalesce(
        try_to_timestamp(trim(regexp_extract({col_name}, '^(.*?) - ', 1)), 'dd/MM/yyyy HH:mm:ss'),
        try_to_timestamp(trim(regexp_extract({col_name}, '^(.*?) - ', 1)), 'dd/MM/yyyy HH:mm')
      )
    """)
def normalize_zone(col_name: str):
    return F.regexp_replace(F.col(col_name), r"BZN\|", "")
def normalize_type_name(s: str) -> str:
    s = (s or "unknown").strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    if not s:
        s = "unknown"
    return f"gen_{s}_mw_avg"
  
normalize_type_udf = F.udf(normalize_type_name)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

table_df = spark.table("h2.raw_entsoe.prices_all_raw")

In [0]:
# Optional: if you have multiple Sequence values and want one canonical record
# table_df = table_df.filter(F.col("Sequence") == "Without Sequence")


In [0]:
df = spark.table("h2.raw_entsoe.prices_all_raw")
df.printSchema()


In [0]:
prices_clean = (
    table_df 
    .withColumn("event_time_local", parse_mtu_start("MTU_CET_CEST"))
    .withColumn("zone", normalize_zone("Area"))
    .withColumn("price_eur_mwh", F.expr("try_cast(Day_ahead_Price_EUR_MWh as double)"))
   .filter(F.col("price_eur_mwh").isNotNull())
   .select(
       F.to_utc_timestamp(F.col("event_time_local"), "Europe/Amsterdam").alias("event_time_utc"),
       "zone",
       "price_eur_mwh",
   )
)
prices_clean.write.mode("overwrite").format('delta').saveAsTable(T_PRICES_CLEAN)

In [0]:
from pyspark.sql import functions as F


load_raw = spark.table("h2.raw_entsoe.load_all_raw")

In [0]:
load_raw.printSchema()

In [0]:
load_clean = (
    load_raw
     .withColumn("event_time_local", parse_mtu_start("MTU_CET_CEST"))
    .withColumn("zone", normalize_zone("Area"))
    .withColumn("actual_total_load_mw", F.expr("try_cast(Actual_Total_Load_MW as double)"))
    .withColumn("day_ahead_total_load_forecast_mw", F.expr("try_cast(Day_Ahead_Total_Load_Forecast_MW as double)"))
     .filter(F.col("event_time_local").isNotNull())
    .withColumn("event_hour_local", F.date_trunc("hour", F.col("event_time_local")))
    .groupBy("event_hour_local", "zone")
    .agg(
       F.avg("actual_total_load_mw").alias("avg_actual_total_load_mw"),
       F.avg("day_ahead_total_load_forecast_mw").alias("day_ahead_total_load_forecast_mw")
       )
    .select(
        F.to_utc_timestamp("event_hour_local","Europe/Amsterdam").alias("event_time_utc"),
        "zone",
        "avg_actual_total_load_mw",
        "day_ahead_total_load_forecast_mw"
    )
    .dropDuplicates(["event_time_utc", "zone"])
 )

load_clean.write.mode("overwrite").format('delta').saveAsTable(T_LOAD_CLEAN)

In [0]:
from pyspark.sql import functions as F
load_clean.selectExpr(
    "count(*) as rows",
    "min(event_time_utc) as min_ts",
    "max(event_time_utc) as max_ts"
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

generation_raw = spark.table("h2.raw_entsoe.generation_all_raw")

In [0]:
generation_by_type = (
    generation_raw
    .withColumn("event_time_local", parse_mtu_start("MTU_CET_CEST"))
    .withColumn("zone", normalize_zone("Area"))
    .withColumn("production_type", F.col("Production_Type"))
    .withColumn("generation_mw", F.expr("try_cast(Generation_MW as double)"))
    .filter(F.col("event_time_local").isNotNull() & F.col("generation_mw").isNotNull())
    .withColumn("event_hour_local", F.date_trunc("hour", F.col("event_time_local")))
    .withColumn("event_time_utc", F.to_utc_timestamp("event_hour_local", "Europe/Amsterdam"))
    .groupBy("event_time_utc", "zone", "production_type")
    .agg(F.avg("generation_mw").alias("generation_mw_avg"))
)
generation_by_type.write.mode("overwrite").format("delta").saveAsTable(T_GEN_BY_TYPE)
# =========================
# 4) GENERATION WIDE (pivot types to columns)
# =========================


In [0]:
base_for_pivot = (
    generation_by_type
    .withColumn("type_col", F.lower(F.col("production_type")))
    .withColumn("type_col", F.regexp_replace(F.col("type_col"), r"[^a-z0-9]+", "_"))
    .withColumn("type_col", F.regexp_replace(F.col("type_col"), r"_+", "_"))
    .withColumn("type_col", F.regexp_replace(F.col("type_col"), r"^_|_$", ""))
    # .withColumn("type_col", F.concat(F.lit("gen_"), F.col("type_col"), F.lit("_mw_avg")))
    .select("event_time_utc", "zone", "type_col", "generation_mw_avg")
)
# 2) pivot on sanitized column names
type_cols = [r["type_col"] for r in base_for_pivot.select("type_col").distinct().collect() if r["type_col"]]
generation_wide = (
    base_for_pivot
    .groupBy("event_time_utc", "zone")
    .pivot("type_col", type_cols)
    .agg(F.first("generation_mw_avg"))
    .fillna(0.0)
)
#total column
if type_cols:
    generation_wide = generation_wide.withColumn(
        "total_generation_mw_avg",
        F.expr(" + ".join([f"`{c}`" for c in type_cols]))
    )
else:
    generation_wide = generation_wide.withColumn("total_generation_mw_avg", F.lit(0.0))

generation_wide.write.mode("overwrite").format("delta").saveAsTable(T_GEN_WIDE)

In [0]:
from pyspark.sql import functions as F

weather_raw = spark.table("h2.raw_era5.weather_all_raw")

In [0]:
weather_raw.printSchema()

In [0]:
weather_clean = (
    weather_raw
    .withColumn("event_time_utc",F.to_timestamp("event_time_utc"))
    .withColumn("zone", normalize_zone("zone"))
    .withColumn("temperature_c", F.expr("try_cast(temperature_c as double)"))
    .withColumn("u10_ms", F.expr("try_cast(u10_ms as double)"))
    .withColumn("v10_ms", F.expr("try_cast(v10_ms as double)"))
    .withColumn("surface_pressure_pa", F.expr("try_cast(surface_pressure_pa as double)"))
    .withColumn("wind_speed_ms", F.expr("sqrt(u10_ms*u10_ms + v10_ms*v10_ms)"))
    .filter(F.col("event_time_utc").isNotNull())
    .select("event_time_utc", "zone", "temperature_c", "u10_ms", "v10_ms", "wind_speed_ms", "surface_pressure_pa")
)

In [0]:
weather_clean.write.mode("overwrite").format('delta').saveAsTable(T_ERA5_CLEAN)
from pyspark.sql import functions as F

weather_clean.selectExpr(
    "count(*) as rows",
    "min(event_time_utc) as min_ts",
    "max(event_time_utc) as max_ts"
).show(truncate=False)

In [0]:
%python
weather_df = spark.table('h2.silver.weather_clean')
load_df = spark.table('h2.silver.entsoe_load_clean')

print('Weather event_time_utc schema:', weather_df.schema['event_time_utc'])
print('ENTSOE event_time_utc schema:', load_df.schema['event_time_utc'])

print('\nSample weather event_time_utc:')
display(weather_df.select('event_time_utc').limit(5))

print('\nSample ENTSOE event_time_utc:')
display(load_df.select('event_time_utc').limit(5))

In [0]:
# %python
# weather_times = spark.table("h2.silver.weather_clean").select("event_time_utc").distinct().limit(10)
# load_times = spark.table("h2.silver.entsoe_load_clean").select("event_time_utc").distinct().limit(10)
# display(weather_times)
# display(load_times)

In [0]:
# Show inner join of weather and ENTSOE load on event_time_utc and zone
weather_df = spark.table('h2.silver.weather_clean')
load_df = spark.table('h2.silver.entsoe_load_clean')

joined = weather_df.join(load_df, ["event_time_utc", "zone"], "inner")
display(joined.limit(10))

In [0]:
stations_raw = spark.table("h2.raw_h2stations.stations_nl_raw")
stations_raw.printSchema()
stations_clean = (
    stations_raw
    .withColumn("id",F.expr("try_cast(id as integer)"))
    .withColumn("country", F.upper(F.col("country")))
    .withColumn("status_id", F.expr("try_cast(status_id as integer)"))
    .withColumn("latitude", F.expr("try_cast(latitude as double)"))
    .withColumn("longitude", F.expr("try_cast(longitude as double)"))
    .withColumn("snapshot_ts_utc", F.to_timestamp("snapshot_ts_utc"))
    .filter(
        (F.col("country") == "NL")&
        (F.col("latitude").between(-90,90)& F.col("longitude").between(-180,180))
    )
    .select("id","station_name","country", "status_id",
            "status_name", "latitude", "longitude", "snapshot_ts_utc")
)

stations_clean.write.mode("overwrite").format("delta").saveAsTable(T_STATIONS_CLEAN)


In [0]:
## 7) DAY 3 INTEGRATION TABLE
prices = spark.table(T_PRICES_CLEAN)
load = spark.table(T_LOAD_CLEAN)
gen_wide = spark.table(T_GEN_WIDE)
weather = spark.table(T_ERA5_CLEAN)
stations = spark.table(T_STATIONS_CLEAN)

In [0]:
station_features = (
    stations.groupBy(F.col("country").alias("zone"))
    .agg(
        F.count('*').alias("h2_station_total"),
        F.sum(F.when(F.col("status_id") == 1, 1).otherwise(0)).alias("h2_station_operational")
    )
    .withColumn(
        "h2_station_active_ratio",
         F.when(F.col("h2_station_total") > 0,
                F.col("h2_station_operational")/F.col("h2_station_total")).otherwise(F.lit(0.0))
    )
)

In [0]:
training_base = (
    prices.alias("p")
    .join(load.alias("l"), ["event_time_utc", "zone"], "inner")
    .join(gen_wide.alias("g"), ["event_time_utc", "zone"], "left")
    .join(weather.alias("w"), ["event_time_utc", "zone"], "left")
    # .join(station_features.alias("s"), ["zone"], "left")
    .withColumn("hour_of_day", F.hour("event_time_utc"))
    .withColumn("day_of_week", F.dayofweek("event_time_utc"))
    .withColumn("month", F.month("event_time_utc"))
)
display(training_base.limit(10))
training_base.write.mode("overwrite").format("delta").saveAsTable(T_TRAINING_BASE)

In [0]:
for t in [
        T_PRICES_CLEAN,
        T_LOAD_CLEAN,
        T_GEN_BY_TYPE,
        T_GEN_WIDE,
        T_ERA5_CLEAN,
        T_STATIONS_CLEAN,
        T_TRAINING_BASE
    ]:
    print(t,spark.table(t).count())

In [0]:
spark.table("h2.silver.entsoe_generation_hourly_wide").printSchema()

In [0]:
from pyspark.sql import functions as F

src_table = "h2.silver.entsoe_generation_hourly_wide"
df = spark.table(src_table)

#identify generation feature columns
gold_dep = (
    df
    .withColumn(
        "renewable_generation_mw",
        F.coalesce(F.col("biomass"), F.lit(0.0)) +
        F.coalesce(F.col("hydro_run_of_river_and_pondage"), F.lit(0.0)) +
        F.coalesce(F.col("wind_onshore"), F.lit(0.0)) +
        F.coalesce(F.col("wind_offshore"), F.lit(0.0)) +
        F.coalesce(F.col("solar"), F.lit(0.0))
    )
    .withColumn(
        "non_renewable_generation_mw",
        F.coalesce(F.col("fossil_hard_coal"), F.lit(0.0)) +
        F.coalesce(F.col("fossil_gas"), F.lit(0.0)) +
        F.coalesce(F.col("nuclear"), F.lit(0.0)) +
        F.coalesce(F.col("waste"), F.lit(0.0)) +
        F.coalesce(F.col("other"), F.lit(0.0))
    )
    .withColumn(
        "total_generation_mw",
        F.coalesce(F.col("total_generation_mw_avg"), F.lit(0.0))
    )
    .withColumn(
        "renewable_share_pct",
        F.when(F.col("total_generation_mw") > 0, F.col("renewable_generation_mw") / F.col("total_generation_mw")).otherwise(F.lit(0.0))
    )
    .withColumn(
        "non_renewable_share_pct",
        F.when(F.col("total_generation_mw") > 0, F.col("non_renewable_generation_mw") / F.col("total_generation_mw")).otherwise(F.lit(0.0))
    )
    .select(
        "event_time_utc",
        "zone",
        "renewable_generation_mw",
        "non_renewable_generation_mw",
        "total_generation_mw",
        "renewable_share_pct",
        "non_renewable_share_pct"
    )
)
# write
gold_dep.write.mode("overwrite").format("delta").saveAsTable("h2.gold.generation_dependency_hourly")


In [0]:
gold_dep.selectExpr(
    "count(*) as rows",
    "avg(renewable_share_pct) as avg_renewable_share",
    "avg(non_renewable_share_pct) as avg_non_renewable_share"
).show()

In [0]:
from pyspark.sql import functions as F
stations = spark.table("h2.silver.h2stations_clean")
station_summary = (
    stations
    .groupBy(F.col("country").alias("zone"))
    .agg(
        F.count("*").alias("stations_total"),
        F.sum(F.when(F.col("status_id") == 1, 1).otherwise(0)).alias("stations_in_operation"),
        F.sum(F.when(F.col("status_id") == 2, 1).otherwise(0)).alias("stations_planned"),
        F.sum(F.when(F.col("status_id") == 3, 1).otherwise(0)).alias("stations_old_projects")
    )
    .withColumn(
        "active_ratio",
        F.when(F.col("stations_total") > 0, F.col("stations_in_operation") / F.col("stations_total")).otherwise(F.lit(0.0))
    )
)
station_summary.write.mode("overwrite").format("delta").saveAsTable("h2.gold.h2_station_status_summary")
display(station_summary)

In [0]:

# 4) h2.gold.renewable_mix_hourly
from pyspark.sql import functions as F
g = spark.table("h2.silver.entsoe_generation_hourly_wide")
renewable_mix = (
    g
    .withColumn("renewable_generation_mw",
        F.coalesce(F.col("biomass"), F.lit(0.0)) +
        F.coalesce(F.col("hydro_run_of_river_and_pondage"), F.lit(0.0)) +
        F.coalesce(F.col("wind_onshore"), F.lit(0.0)) +
        F.coalesce(F.col("wind_offshore"), F.lit(0.0)) +
        F.coalesce(F.col("solar"), F.lit(0.0))
    )
    .withColumn("total_generation_mw", F.coalesce(F.col("total_generation_mw_avg"), F.lit(0.0)))
    .withColumn(
        "renewable_share_pct",
        F.when(F.col("total_generation_mw") > 0, F.col("renewable_generation_mw") / F.col("total_generation_mw")).otherwise(F.lit(0.0))
    )
    .select(
        "event_time_utc",
        "zone",
        F.coalesce(F.col("biomass"), F.lit(0.0)).alias("biomass_mw"),
        F.coalesce(F.col("hydro_run_of_river_and_pondage"), F.lit(0.0)).alias("hydro_mw"),
        F.coalesce(F.col("wind_onshore"), F.lit(0.0)).alias("wind_onshore_mw"),
        F.coalesce(F.col("wind_offshore"), F.lit(0.0)).alias("wind_offshore_mw"),
        F.coalesce(F.col("solar"), F.lit(0.0)).alias("solar_mw"),
        "renewable_generation_mw",
        "total_generation_mw",
        "renewable_share_pct"
    )
)
display(renewable_mix)
renewable_mix.write.mode("overwrite").format("delta").saveAsTable("h2.gold.renewable_mix_hourly")

In [0]:
prices = spark.table("h2.silver.entsoe_prices_clean")
load = spark.table("h2.silver.entsoe_load_clean")
gen_wide = spark.table("h2.silver.entsoe_generation_hourly_wide")
weather = spark.table("h2.silver.weather_clean")
market_weather_hourly = (
    prices.alias("p")
    .join(load.alias("l"), ["event_time_utc", "zone"], "inner")
    .join(gen_wide.alias("g"), ["event_time_utc", "zone"], "left")
    .join(weather.alias("w"), ["event_time_utc", "zone"], "left")
    .withColumn("hour_of_day", F.hour("event_time_utc"))
    .withColumn("day_of_week", F.dayofweek("event_time_utc"))
    .withColumn("month", F.month("event_time_utc"))
)
market_weather_hourly.write.mode("overwrite").format("delta").saveAsTable("h2.gold.market_weather_hourly")

In [0]:

# 5) h2.gold.fossil_dependency_hourly

from pyspark.sql import functions as F
g = spark.table("h2.silver.entsoe_generation_hourly_wide")
fossil_dependency = (
    g
    .withColumn("fossil_generation_mw",
        F.coalesce(F.col("fossil_gas"), F.lit(0.0)) +
        F.coalesce(F.col("fossil_hard_coal"), F.lit(0.0))
    )
    .withColumn("non_renewable_generation_mw",
        F.coalesce(F.col("fossil_gas"), F.lit(0.0)) +
        F.coalesce(F.col("fossil_hard_coal"), F.lit(0.0)) +
        F.coalesce(F.col("nuclear"), F.lit(0.0)) +
        F.coalesce(F.col("waste"), F.lit(0.0)) +
        F.coalesce(F.col("other"), F.lit(0.0))
    )
    .withColumn("total_generation_mw", F.coalesce(F.col("total_generation_mw_avg"), F.lit(0.0)))
    .withColumn(
        "fossil_share_pct",
        F.when(F.col("total_generation_mw") > 0, F.col("fossil_generation_mw") / F.col("total_generation_mw")).otherwise(F.lit(0.0))
    )
    .withColumn(
        "non_renewable_share_pct",
        F.when(F.col("total_generation_mw") > 0, F.col("non_renewable_generation_mw") / F.col("total_generation_mw")).otherwise(F.lit(0.0))
    )
    .select(
        "event_time_utc",
        "zone",
        F.coalesce(F.col("fossil_gas"), F.lit(0.0)).alias("fossil_gas_mw"),
        F.coalesce(F.col("fossil_hard_coal"), F.lit(0.0)).alias("fossil_hard_coal_mw"),
        "fossil_generation_mw",
        "non_renewable_generation_mw",
        "total_generation_mw",
        "fossil_share_pct",
        "non_renewable_share_pct"
    )
)
fossil_dependency.write.mode("overwrite").format("delta").saveAsTable("h2.gold.fossil_dependency_hourly")

In [0]:
# 6) h2.gold.daily_ops_kpis
from pyspark.sql import functions as F
mw = spark.table("h2.gold.market_weather_hourly")
dep = spark.table("h2.gold.generation_dependency_hourly")
st = spark.table("h2.gold.h2_station_status_summary")
daily_ops_kpis = (
    mw.alias("m")
    .join(dep.alias("d"), ["event_time_utc", "zone"], "left")
    .withColumn("event_date", F.to_date("event_time_utc"))
    .groupBy("event_date", "zone")
    .agg(
        F.avg("price_eur_mwh").alias("avg_price_eur_mwh"),
        F.max("avg_actual_total_load_mw").alias("peak_load_mw"),
        F.avg("renewable_share_pct").alias("avg_renewable_share_pct"),
        F.avg("non_renewable_share_pct").alias("avg_non_renewable_share_pct"),
        F.avg("temperature_c").alias("avg_temperature_c"),
        F.avg("wind_speed_ms").alias("avg_wind_speed_ms")
    )
    .join(st.alias("s"), ["zone"], "left")
    .select(
        "event_date",
        "zone",
        "avg_price_eur_mwh",
        "peak_load_mw",
        "avg_renewable_share_pct",
        "avg_non_renewable_share_pct",
        "avg_temperature_c",
        "avg_wind_speed_ms",
        "stations_total",
        "stations_in_operation",
        "active_ratio"
    )
)

daily_ops_kpis.write.mode("overwrite").format("delta").saveAsTable("h2.gold.daily_ops_kpis")

In [0]:
# 7) h2.gold.model_scoring_base
from pyspark.sql import functions as F
mw = spark.table("h2.gold.market_weather_hourly")
dep = spark.table("h2.gold.generation_dependency_hourly")
model_scoring_base = (
    mw.alias("m")
    .join(dep.alias("d"), ["event_time_utc", "zone"], "left")
    .select(
        "event_time_utc",
        "zone",
        "price_eur_mwh",
        "avg_actual_total_load_mw",
        "day_ahead_total_load_forecast_mw",
        "temperature_c",
        "u10_ms",
        "v10_ms",
        "wind_speed_ms",
        "surface_pressure_pa",
        "renewable_generation_mw",
        "non_renewable_generation_mw",
        "renewable_share_pct",
        "non_renewable_share_pct",
        "hour_of_day",
        "day_of_week",
        "month"
    )
)
model_scoring_base.write.mode("overwrite").format("delta").saveAsTable("h2.gold.model_scoring_base")

In [0]:
# # Run log table for ops
# spark.sql("""
# CREATE SCHEMA IF NOT EXISTS h2.ops
# """)
# spark.sql("""
# CREATE TABLE IF NOT EXISTS h2.ops.pipeline_run_log (
#   run_ts_utc TIMESTAMP,
#   stage STRING,
#   status STRING,
#   row_count BIGINT
# ) USING DELTA
# """)
# rows = spark.table("h2.silver.h2_training_base").count()
# spark.sql(f"""
# INSERT INTO h2.ops.pipeline_run_log
# SELECT current_timestamp(), 'day4_gold_build', 'success', {rows}
# """)